# Data-driven optimal weights (K=3 / K=4 / K=5)

Fit **static** GMM on the training window **1971-03-31 → 1990-12-31** only, then solve
long-only mean-variance portfolios on the **three investable sleeves** for each regime.

- Regime detection uses all **17 factors** (same feature matrix as the GMM pipeline).
- Portfolio weights are optimized on **SPXT / LUATTRUU / BCOMTR** only.
- Weights are **frozen** after training and reused in backtests from 1990 onward.

**No look-ahead:** training ends 1990-12-31; post-1990 regime labels come from the
existing walk-forward CSVs (`walk_forward_k3/4/5.csv`).

In [1]:
import sys
from pathlib import Path

import pandas as pd


def _find_pmr_root() -> Path:
    start = Path.cwd().resolve()
    for parent in [start, *start.parents]:
        if (parent / "scripts" / "paths.py").is_file():
            return parent
    raise FileNotFoundError("Run from inside pmr_paper/")


PROJECT_ROOT = _find_pmr_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.paths import OUTPUT_DIR, load_features
from scripts.portfolio_allocation import (
    DEFAULT_TRAIN_END,
    DEFAULT_TRAIN_START,
    build_regime_portfolios,
    save_regime_portfolios,
)

features = load_features()
print(f"Features: {features.shape} ({features.index.min().date()} → {features.index.max().date()})")
print(f"Training window: {DEFAULT_TRAIN_START} → {DEFAULT_TRAIN_END}")

Features: (663, 17) (1971-03-31 → 2026-05-31)
Training window: 1971-03-31 → 1990-12-31


## Fit static GMM + MV weights per regime

For each K ∈ {3, 4, 5}:
1. Fit static GMM on 1971–1990 training slice (17 factors).
2. Map sklearn components → economic regime ids (Hungarian labeling).
3. Extract μₖ, Σₖ per regime; subset to 3 investable assets.
4. Solve long-only max-Sharpe: wₖ* with w ≥ 0, Σw = 1.
5. Save frozen weights to `data/outputs/data_driven_portfolios_k{k}.csv`.

In [2]:
PORTFOLIO_TABLES = {}
SAVED_PATHS = {}

for k in (3, 4, 5):
    table = build_regime_portfolios(features, k)
    path = save_regime_portfolios(table, k)
    PORTFOLIO_TABLES[k] = table
    SAVED_PATHS[k] = path
    print(f"\n=== K={k} — saved to {path} ===")
    display(table.round(3))


=== K=3 — saved to /Users/antoinea/Desktop/Master_Thesis/data/outputs/data_driven_portfolios_k3.csv ===


,regime_id,regime_name,w_SPXT,w_LUATTRUU,w_BCOMTR
0,0,Bear,0.000,0.000,1.000
1,1,Neutral,0.210,0.501,0.289
2,2,Bull,0.084,0.759,0.156



=== K=4 — saved to /Users/antoinea/Desktop/Master_Thesis/data/outputs/data_driven_portfolios_k4.csv ===


,regime_id,regime_name,w_SPXT,w_LUATTRUU,w_BCOMTR
0,0,Crisis,0.000,0.705,0.295
1,1,Inflation,0.000,0.824,0.176
2,2,Steady State,0.111,0.673,0.216
3,3,Walking on Ice,0.427,0.573,0.000



=== K=5 — saved to /Users/antoinea/Desktop/Master_Thesis/data/outputs/data_driven_portfolios_k5.csv ===


,regime_id,regime_name,w_SPXT,w_LUATTRUU,w_BCOMTR
0,0,Crisis,0.000,0.132,0.868
1,1,Inflation,0.000,1.000,0.000
2,2,Steady State,0.856,0.121,0.023
3,3,Walking on Ice,0.597,0.398,0.005
4,4,Bull Market,0.342,0.409,0.249


## 14-asset portfolios (TRS-investable factors)

Supervisor-approved investable set: all `features.csv` columns **except** VIX, USGG3M, LUACOAS.
Saved to `data/outputs/data_driven_portfolios_k{k}_14.csv`.

In [3]:
from scripts.portfolio_allocation import INVESTABLE_14_COLS, save_regime_portfolios

TABLES_14 = {}
SAVED_PATHS_14 = {}

for k in (3, 4, 5):
    table = build_regime_portfolios(features, k, investable_cols=INVESTABLE_14_COLS)
    path = save_regime_portfolios(table, k, variant="14")
    TABLES_14[k] = table
    SAVED_PATHS_14[k] = path
    print(f"\n=== K={k} — 14 assets — saved to {path} ===")
    display(table.round(3))


=== K=3 — 14 assets — saved to /Users/antoinea/Desktop/Master_Thesis/data/outputs/data_driven_portfolios_k3_14.csv ===


,regime_id,regime_name,w_SPXT,w_LUATTRUU,w_LF98TRUU,w_BCOMTR,w_MXEF,w_BCIT1T,w_DXY,w_NEIXCTAT,w_M1WOSC,w_M1WO000V,w_M1WOMOM,w_M1WOQU,w_M1WOMVOL,w_DBFXCARU
0,0,Bear,0.0,0.000,0.0,0.240,0.000,0.096,0.665,0.000,0.0,0.0,0.000,0.0,0.0,0.000
1,1,Neutral,0.0,0.133,0.0,0.125,0.031,0.000,0.000,0.112,0.0,0.0,0.037,0.0,0.0,0.562
2,2,Bull,0.0,0.183,0.0,0.042,0.000,0.001,0.064,0.000,0.0,0.0,0.003,0.0,0.0,0.709



=== K=4 — 14 assets — saved to /Users/antoinea/Desktop/Master_Thesis/data/outputs/data_driven_portfolios_k4_14.csv ===


,regime_id,regime_name,w_SPXT,w_LUATTRUU,w_LF98TRUU,w_BCOMTR,w_MXEF,w_BCIT1T,w_DXY,w_NEIXCTAT,w_M1WOSC,w_M1WO000V,w_M1WOMOM,w_M1WOQU,w_M1WOMVOL,w_DBFXCARU
0,0,Crisis,0.0,0.518,0.000,0.000,0.000,0.093,0.000,0.389,0.000,0.000,0.000,0.0,0.0,0.000
1,1,Inflation,0.0,0.167,0.000,0.033,0.000,0.000,0.109,0.000,0.000,0.000,0.000,0.0,0.0,0.691
2,2,Steady State,0.0,0.227,0.098,0.076,0.039,0.006,0.000,0.070,0.000,0.000,0.016,0.0,0.0,0.469
3,3,Walking on Ice,0.0,0.229,0.000,0.010,0.000,0.000,0.000,0.062,0.131,0.092,0.131,0.0,0.0,0.345



=== K=5 — 14 assets — saved to /Users/antoinea/Desktop/Master_Thesis/data/outputs/data_driven_portfolios_k5_14.csv ===


,regime_id,regime_name,w_SPXT,w_LUATTRUU,w_LF98TRUU,w_BCOMTR,w_MXEF,w_BCIT1T,w_DXY,w_NEIXCTAT,w_M1WOSC,w_M1WO000V,w_M1WOMOM,w_M1WOQU,w_M1WOMVOL,w_DBFXCARU
0,0,Crisis,0.000,0.046,0.000,0.301,0.000,0.068,0.000,0.586,0.000,0.000,0.000,0.000,0.000,0.000
1,1,Inflation,0.000,0.508,0.000,0.000,0.000,0.000,0.432,0.000,0.000,0.000,0.000,0.000,0.000,0.060
2,2,Steady State,0.145,0.021,0.021,0.004,0.026,0.000,0.000,0.007,0.037,0.074,0.018,0.119,0.131,0.397
3,3,Walking on Ice,0.096,0.064,0.029,0.001,0.046,0.000,0.000,0.000,0.061,0.073,0.054,0.114,0.094,0.369
4,4,Bull Market,0.034,0.041,0.019,0.025,0.029,0.001,0.000,0.027,0.033,0.068,0.028,0.087,0.054,0.554


## Summary

These CSVs are consumed by `data_driven_3`, `data_driven_4`, and `data_driven_5` in
`01_strategy_comparison.ipynb`. The 14-asset variants (`data_driven_3/4/5_14`) use `*_14.csv`.
Each strategy switches to the frozen regime portfolio using walk-forward hard labels
(or probability blends) for the matching K.